# 使用 Meta 家族模型構建

## 介紹

本課程將涵蓋：

- 探索兩個主要的 Meta 家族模型 - Llama 3.1 和 Llama 3.2
- 了解每個模型的使用案例和場景
- 代碼範例展示每個模型的獨特功能


## Meta 家族模型

在本課程中，我們將探索 Meta 家族或「Llama Herd」中的兩個模型 - Llama 3.1 和 Llama 3.2

這些模型有不同的變體，可在 [Microsoft Foundry Models catalog](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) 中找到。

> **注意：** GitHub Models 將於 2026 年 7 月底退休。這裡有更多關於使用 [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) 進行 AI 模型原型設計的詳細資訊。

模型變體：
- Llama 3.1 - 70B 指令型
- Llama 3.1 - 405B 指令型
- Llama 3.2 - 11B 視覺指令型
- Llama 3.2 - 90B 視覺指令型

*注意：Llama 3 也可以在 Microsoft Foundry Models 找到，但本課程不涵蓋*



## Llama 3.1 

Llama 3.1 拥有 4050 億個參數，屬於開源 LLM 範疇。 

該版本是對早期發佈的 Llama 3 的升級，提供： 

- 更大的上下文窗口 - 128k 代幣，而非 8k 代幣 
- 更大的最大輸出代幣數 - 4096 對比 2048 
- 更好的多語言支持 - 由於訓練代幣數增加 

這使 Llama 3.1 能處理更複雜的用例，在構建生成式 AI 應用時包括： 
- 原生函數調用 - 能夠在 LLM 工作流程之外調用外部工具和函數
- 更佳的 RAG 性能 - 由於更高的上下文窗口 
- 合成數據生成 - 能產生有效數據來支持微調等任務 



### 原生函數調用 

Llama 3.1 已經過微調，以更有效地進行函數或工具調用。它還內建了兩個工具，模型可以根據使用者的提示識別需要使用這些工具。這些工具是： 

- **Brave Search** - 可用於透過網絡搜索獲取最新資訊，例如天氣 
- **Wolfram Alpha** - 可用於更複雜的數學計算，因此無需編寫自己的函數。 

你也可以建立自己的自訂工具，讓大語言模型調用。 

在以下程式碼範例中： 

- 我們在系統提示中定義可用工具（brave_search、wolfram_alpha）。 
- 發送一個詢問某個城市天氣的使用者提示。 
- 大語言模型將會回應對 Brave Search 工具的調用，形式如下 `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*注意：此範例僅進行工具調用，若想獲得結果，你需要在 Brave API 頁面建立免費帳戶並定義該函數本身` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

儘管作為一個大型語言模型，Llama 3.1 有一個限制是多模態能力。也就是能夠使用不同類型的輸入，如圖像作為提示並提供回應。這種能力是 Llama 3.2 的主要功能之一。這些功能還包括： 

- 多模態 - 能夠評估文本和圖像提示 
- 小到中等大小的變體（11B 和 90B）- 提供靈活的部署選項， 
- 僅文本變體（1B 和 3B）- 允許模型部署於邊緣/行動裝置並提供低延遲 

多模態支持在開源模型領域是一大進展。以下代碼範例同時使用圖像和文本提示，從 Llama 3.2 90B 獲得對圖像的分析。 

### 使用 Llama 3.2 的多模態支持


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## 學習不止於此，繼續你的旅程

完成本課程後，請查看我們的 [生成式人工智能學習系列](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst)，繼續提升你的生成式人工智能知識！


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
